# micro-sam GPU segmentation in napari

In [3]:
import os
import sys
_qt_plugins = os.path.join(sys.prefix, "Library", "lib", "qt6", "plugins")
if os.path.isdir(_qt_plugins):
    os.environ["QT_PLUGIN_PATH"] = _qt_plugins
    os.environ["QT_QPA_PLATFORM_PLUGIN_PATH"] = os.path.join(_qt_plugins, "platforms")
else:
    print("WARNING: Qt6 plugin dir not found at", _qt_plugins)
import warnings
warnings.filterwarnings("ignore")
import torch
import napari
import micro_sam
from pathlib import Path
from micro_sam.training import train_sam
from micro_sam.util import export_custom_sam_model
import matplotlib.pyplot as plt
from skimage.color import label2rgb
import numpy as np
import imageio.v3 as imageio
from magicgui.widgets import Container, PushButton, Label
import random
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation
import imageio.v3 as imageio
from torch.utils.data import random_split
import torch_em
from torch_em.data import MinInstanceSampler
from micro_sam.training import train_sam_for_configuration, default_sam_dataset
from micro_sam.training.util import get_raw_transform
from micro_sam.sam_annotator import image_folder_annotator

print("micro_sam:", micro_sam.__version__)
print("napari   :", napari.__version__)
print("torch    :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device :", torch.cuda.get_device_name(0) if device == "cuda" else "cpu")


micro_sam: 1.8.7
napari   : 0.8.0
torch    : 2.12.1
CUDA available: True
Using device : NVIDIA GeForce RTX 5080


# Fine Tuning the Model
- fine-tune the Light Microscopy model (vit_b_lm) on your seed labels.

In [8]:

data_root = Path(r"C:\Users\taylorhearn\Desktop")
dataset_dirs = sorted(
    d for d in data_root.iterdir()
    if d.is_dir() and (d / "images").is_dir() and (d / "labels").is_dir()
)
print(f"found {len(dataset_dirs)} dataset folders:", [d.name for d in dataset_dirs])

patch_shape = (512, 512)
# Checkpoints + logs are written to: <save_root>/checkpoints/<name>/best.pt
save_root = r"Z:\Bel\Dorota_Image_Analzsis_Size_Spaces\C1_Dapi_C2_VeCad3"
name = "round1_curation"

# Match each image to its label by sorted filename order.
image_paths, label_paths = [], []
for d in dataset_dirs:
    image_paths += sorted((d / "images").glob("*.tif"))
    label_paths += sorted((d / "labels").glob("*.tif"))

print(f"images: {len(image_paths)}  |  labels: {len(label_paths)}")
assert len(image_paths) > 0, "No images found - check the folder / pattern."
assert len(image_paths) == len(label_paths), "image/label counts differ - names must correspond."


def to_channels_first(a):
    """RGB/multichannel image (H, W, C) -> (C, H, W); 2D grayscale is left unchanged."""
    a = np.asarray(a)
    if a.ndim == 3:
        a = np.moveaxis(a, -1, 0)
    return a


# RGB images -> (C, H, W) so the 3 channels are a channel axis, not a spatial one. Labels stay 2D (H, W).
raw_arrays = [to_channels_first(imageio.imread(p)).astype(np.int32) for p in image_paths]
label_arrays = [imageio.imread(p).astype(np.int32) for p in label_paths]
print("example image shape:", raw_arrays[0].shape, "| example label shape:", label_arrays[0].shape)
print("raw range:", int(np.min(raw_arrays[0])), "-", int(np.max(raw_arrays[0])))

raw_transform = get_raw_transform("normalize_percentile")

# with_channels=True feeds the 3 RGB channels to the model (raw is (C, H, W)); labels remain (H, W).
# min_num_instances=1 keeps single-object images.
ds = default_sam_dataset(
    raw_paths=raw_arrays, raw_key=None,
    label_paths=label_arrays, label_key=None,
    patch_shape=patch_shape,
    with_segmentation_decoder=True,
    with_channels=True,
    raw_transform=raw_transform,
    sampler=MinInstanceSampler(min_num_instances=1, min_size=25),
)
n_val = max(1, int(0.1 * len(ds)))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])

# num_workers=0 on Windows notebooks avoids multiprocessing/spawn issues.
train_loader = torch_em.get_data_loader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader = torch_em.get_data_loader(val_ds, batch_size=1, shuffle=True, num_workers=0)
print(f"train patches/epoch: {len(train_ds)}  |  val: {len(val_ds)}")



found 1 dataset folders: ['COPY_CAN_DELETE_Dorota_Akinola']
images: 9  |  labels: 0


AssertionError: image/label counts differ - names must correspond.

In [4]:
# Drop only truly-empty label images (0 objects); single-object images are kept, then rebuild loaders.
def _n_instances(lab):
    return int(np.count_nonzero(np.unique(lab)))   # distinct non-zero label ids

_counts  = [_n_instances(lab) for lab in label_arrays]
_keep    = [i for i, c in enumerate(_counts) if c >= 1]
_dropped = [(image_paths[i].name, _counts[i]) for i in range(len(label_arrays)) if i not in _keep]
if _dropped:
    print(f"dropping {len(_dropped)} image(s) with no labelled objects:")
    for _n, _c in _dropped:
        print(f"   {_n}  ({_c} objects)")
else:
    print("no empty label images found - all pairs kept")

raw_arrays   = [raw_arrays[i]   for i in _keep]
label_arrays = [label_arrays[i] for i in _keep]
assert raw_arrays, "All images were dropped - check that your label files contain instance labels."
print("kept", len(raw_arrays), "image/label pairs")

ds = default_sam_dataset(
    raw_paths=raw_arrays, raw_key=None,
    label_paths=label_arrays, label_key=None,
    patch_shape=patch_shape,
    with_segmentation_decoder=True,
    with_channels=True,
    raw_transform=raw_transform,
    sampler=MinInstanceSampler(min_num_instances=1, min_size=25),
)
n_val = max(1, int(0.1 * len(ds)))
train_ds, val_ds = random_split(ds, [len(ds) - n_val, n_val])
train_loader = torch_em.get_data_loader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader   = torch_em.get_data_loader(val_ds,   batch_size=1, shuffle=True, num_workers=0)
print(f"train patches/epoch: {len(train_ds)}  |  val: {len(val_ds)}")


no empty label images found - all pairs kept
kept 6 image/label pairs
train patches/epoch: 90  |  val: 10


In [5]:

# Retry-on-empaty: if augmentation empties the instance channel, redraw a new patch.
_DsType = type(ds)
if not getattr(_DsType, "_skip_empty_patched", False):
    _orig_getitem = _DsType.__getitem__

    def _getitem_skip_empty(self, index, _orig=_orig_getitem):
        for _ in range(25):
            x, y = _orig(self, index)
            if np.count_nonzero(np.asarray(y)[0]) > 0:   # channel 0 = instance IDs
                return x, y
            index = np.random.randint(len(self))          # try a different patch
        return x, y                                        # give up (astronomically rare)

    _DsType.__getitem__ = _getitem_skip_empty
    _DsType._skip_empty_patched = True
    print("patched: empty augmented patches will now be skipped")
else:
    print("already patched")

patched: empty augmented patches will now be skipped


In [6]:
# Run fine-tuning.
train_sam(
    name=name,
    model_type="vit_b_lm",          # start from the Light Microscopy model
    train_loader=train_loader,
    val_loader=val_loader,
    n_objects_per_batch=10,
    with_segmentation_decoder=True,
    save_root=str(save_root),
    n_epochs=100,                   # early stopping halts sooner once val stops improving
)
print("best checkpoint:", Path(save_root) / "checkpoints" / name / "best.pt")

Verifying labels in 'val' dataloader:  20%|██        | 10/50 [00:01<00:04,  9.42it/s]


Start fitting for 9000 iterations /  100 epochs
with 90 iterations per epoch
Training with mixed precision


Epoch 29: average [s/it]: 13.270799, current metric: 0.079257, best metric: 0.046734:  30%|███       | 2700/9000 [11:05:03<25:51:47, 14.78s/it]

Stopping training because there has been no improvement for 10 epochs
Finished training after 29 epochs / 2700 iterations.
The best epoch is number 18.
Training took 39911.59874820709 seconds (= 11:665:12 hours)
best checkpoint: Z:\Bel\Dorota_Image_Analzsis_Size_Spaces\C1_Dapi_C2_VeCad3\checkpoints\round1_curation\best.pt


In [7]:
# Export the fine-tuned model so it works in the napari GUI / CLI / library.
# In the GUI, put this path in Embedding Settings -> "custom weights path"
# (keep Model = Light Microscopy, size = base).

# Automatically grab the best checkpoint from the run we just trained and export into save_root.
best_ckpt = Path(save_root) / "checkpoints" / name / "best.pt"
assert best_ckpt.is_file(), f"best checkpoint not found: {best_ckpt}"

finetuned_path = Path(save_root) / f"{name}_finetuned.pth"

export_custom_sam_model(
    checkpoint_path=str(best_ckpt),
    model_type="vit_b",              # architecture size only (base); the LM weights are inside best.pt
    save_path=str(finetuned_path),
    with_segmentation_decoder=True,  # keep the decoder so AIS/APG work
)
print("best checkpoint:", best_ckpt)
print("finetuned model exported to:", finetuned_path)


best checkpoint: Z:\Bel\Dorota_Image_Analzsis_Size_Spaces\C1_Dapi_C2_VeCad3\checkpoints\round1_curation\best.pt
finetuned model exported to: Z:\Bel\Dorota_Image_Analzsis_Size_Spaces\C1_Dapi_C2_VeCad3\round1_curation_finetuned.pth
